# 📦 Notebook 01 — Preprocessing (Data Engineering)
### Smart Retail Shelf — IVP + AI Team Project

**Owner role:** Data Engineer

This notebook builds the two preprocessing pipelines that the rest of the
project depends on:

1. **Vision Pipeline** — load/generate a shelf image, resize to `640x640`,
   normalize to `[0, 1]`, and apply `albumentations` augmentation
   (horizontal flip + brightness jitter). Original vs. augmented image are
   shown side-by-side.
2. **AI Pipeline** — generate a mock list of retail "shopping basket"
   transactions, clean nulls, one-hot encode them with
   `mlxtend.preprocessing.TransactionEncoder`, and export the binary matrix
   to `transactions_encoded.csv` for Notebook 03 (`ai_correlation_knn.ipynb`).

> Runs **out-of-the-box** in Google Colab — no external files needed. All
> data is generated synthetically (mock shelf image + mock transactions) so
> the notebook never errors on a missing dataset.


## 0. Setup — Install & Import Dependencies

In [ ]:
# ============================================================
# 0.1  Install required libraries (Colab-safe: quiet, pinned-free)
# ============================================================
!pip install -q opencv-python-headless albumentations mlxtend pandas numpy matplotlib


In [ ]:
# ============================================================
# 0.2  Imports
# ============================================================
import cv2                     # OpenCV -> image I/O, resize, color ops
import numpy as np             # numeric arrays / pixel manipulation
import pandas as pd            # transaction dataframe handling
import matplotlib.pyplot as plt

import albumentations as A     # augmentation pipeline (flip + brightness)

from mlxtend.preprocessing import TransactionEncoder  # one-hot encode baskets

import os
import random

# Reproducibility across the whole notebook
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("All libraries imported successfully ✅")


## 1. Vision Pipeline
### 1.1 Generate / Load a Dummy "Shelf" Image

In a real pipeline this would be `cv2.imread("shelf_photo.jpg")`. Since this
notebook must run standalone in Colab with **zero external files**, we
synthesize a mock retail-shelf image (colored "product" rectangles on a
shelf-like background) using OpenCV drawing primitives. Swap in
`cv2.imread(path)` when real shelf photos are available.


In [ ]:
# ============================================================
# 1.1  Create a synthetic "retail shelf" image with OpenCV
#      (stand-in for cv2.imread("shelf.jpg") in production)
# ============================================================
def generate_dummy_shelf_image(width=800, height=600, n_products=8):
    """
    Builds a mock shelf image: a light background with a wooden-brown
    shelf strip and colored rectangles representing products.
    Returns a BGR uint8 image (OpenCV's native format).
    """
    img = np.full((height, width, 3), (235, 235, 235), dtype=np.uint8)  # light backdrop

    # Draw a "shelf" horizontal bar near the bottom third
    shelf_y = int(height * 0.7)
    cv2.rectangle(img, (0, shelf_y), (width, shelf_y + 25), (60, 110, 160), thickness=-1)  # BGR brown

    # Scatter colorful "product" boxes sitting on the shelf line
    rng = np.random.default_rng(SEED)
    colors = [(66, 133, 244), (52, 168, 83), (234, 67, 53), (251, 188, 5),
              (156, 39, 176), (0, 172, 193), (255, 87, 34), (121, 85, 72)]
    box_w, gap = width // (n_products + 2), 12
    x = gap
    for i in range(n_products):
        h = rng.integers(90, 170)
        y2 = shelf_y
        y1 = y2 - h
        color = colors[i % len(colors)]
        cv2.rectangle(img, (x, y1), (x + box_w - gap, y2), color, thickness=-1)
        cv2.rectangle(img, (x, y1), (x + box_w - gap, y2), (30, 30, 30), thickness=2)  # outline
        x += box_w
    return img

original_image = generate_dummy_shelf_image()
print("Dummy shelf image generated. Shape (H, W, C):", original_image.shape)

# --- If you have a REAL image uploaded to Colab, use this instead: -----
# from google.colab import files
# uploaded = files.upload()
# path = list(uploaded.keys())[0]
# original_image = cv2.imread(path)
# -------------------------------------------------------------------------


### 1.2 Resize to 640×640 and Normalize Pixel Values to [0, 1]

In [ ]:
# ============================================================
# 1.2  Resize -> 640x640 (YOLOv8's default input size) + normalize
# ============================================================
TARGET_SIZE = (640, 640)  # (width, height) expected by cv2.resize

def resize_and_normalize(img_bgr, size=TARGET_SIZE):
    """
    1. Resize image to the target (W, H) using INTER_AREA (good for downscaling).
    2. Convert BGR -> RGB (matplotlib / most DL frameworks expect RGB).
    3. Normalize uint8 [0,255] pixel values to float32 [0,1].
    """
    resized_bgr = cv2.resize(img_bgr, size, interpolation=cv2.INTER_AREA)
    resized_rgb = cv2.cvtColor(resized_bgr, cv2.COLOR_BGR2RGB)
    normalized = resized_rgb.astype(np.float32) / 255.0
    return resized_rgb, normalized

resized_rgb, normalized_img = resize_and_normalize(original_image)

print("Resized shape :", resized_rgb.shape)
print("Pixel range after normalization -> min: %.3f  max: %.3f"
      % (normalized_img.min(), normalized_img.max()))


### 1.3 Augmentation with `albumentations` (Horizontal Flip + Brightness Jitter)

In [ ]:
# ============================================================
# 1.3  Albumentations augmentation pipeline
#      HorizontalFlip -> mirrors the shelf (simulates camera angle variety)
#      RandomBrightnessContrast -> simulates different store lighting
# ============================================================
augment_pipeline = A.Compose([
    A.HorizontalFlip(p=1.0),                                   # always flip for demo visibility
    A.RandomBrightnessContrast(brightness_limit=0.25,
                                contrast_limit=0.15, p=1.0),    # lighting variation
])

# Albumentations expects uint8 RGB input (not the [0,1] float we made above),
# so we augment the resized RGB image, THEN normalize the augmented result.
augmented_uint8 = augment_pipeline(image=resized_rgb)["image"]
augmented_normalized = augmented_uint8.astype(np.float32) / 255.0

print("Augmented image shape:", augmented_uint8.shape)
print("Augmented pixel range -> min: %.3f  max: %.3f"
      % (augmented_normalized.min(), augmented_normalized.max()))


### 1.4 Visualize: Original vs. Preprocessed (Resized + Augmented) Side-by-Side

In [ ]:
# ============================================================
# 1.4  Side-by-side visualization
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Original\n{original_image.shape[1]}x{original_image.shape[0]}")
axes[0].axis("off")

axes[1].imshow(resized_rgb)
axes[1].set_title(f"Resized + Normalized\n{TARGET_SIZE[0]}x{TARGET_SIZE[1]}, [0,1] range")
axes[1].axis("off")

axes[2].imshow(augmented_uint8)
axes[2].set_title("Augmented\n(H-Flip + Brightness Jitter)")
axes[2].axis("off")

plt.suptitle("Vision Preprocessing Pipeline — Smart Retail Shelf", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## 2. AI Pipeline — Transaction Data Preparation
### 2.1 Generate a Dummy "Shopping Basket" Transaction Dataset

Each row simulates one customer's basket (list of items purchased together).
We deliberately inject a few `None` / missing values to demonstrate the
cleaning step.


In [ ]:
# ============================================================
# 2.1  Mock transactional data: list-of-items per customer purchase
#      (stand-in for real POS / checkout data)
# ============================================================
PRODUCT_CATALOG = [
    "Milk", "Bread", "Eggs", "Butter", "Cheese", "Yogurt",
    "Chips", "Soda", "Cereal", "Coffee", "Tea", "Sugar",
    "Rice", "Pasta", "Tomato Sauce", "Chicken", "Frozen Pizza", "Ice Cream"
]

def generate_dummy_transactions(n_transactions=40, seed=SEED):
    """
    Simulates realistic co-purchase behaviour by biasing certain items to
    co-occur (e.g., Bread+Butter, Pasta+Tomato Sauce, Chips+Soda) so that
    the Apriori step in Notebook 03 produces meaningful, non-trivial rules.
    """
    rng = np.random.default_rng(seed)
    affinity_sets = [
        ["Bread", "Butter", "Milk", "Eggs"],
        ["Pasta", "Tomato Sauce", "Cheese"],
        ["Chips", "Soda", "Ice Cream"],
        ["Coffee", "Tea", "Sugar"],
        ["Chicken", "Rice", "Frozen Pizza"],
    ]
    transactions = []
    for _ in range(n_transactions):
        basket = set()
        # Pick 1-2 "affinity" groups (by INDEX, since the groups have
        # different lengths and numpy can't build a ragged array to
        # sample from directly).
        group_idxs = rng.choice(len(affinity_sets), size=rng.integers(1, 3), replace=False)
        for gi in group_idxs:
            group = affinity_sets[gi]
            keep = rng.choice(group, size=rng.integers(2, len(group) + 1), replace=False)
            basket.update(keep.tolist())
        # Sprinkle in a couple of random extra items
        n_extra = rng.integers(0, 3)
        if n_extra > 0:
            extras = rng.choice(PRODUCT_CATALOG, size=n_extra, replace=False)
            basket.update(extras.tolist())
        transactions.append(sorted(basket))
    return transactions

raw_transactions = generate_dummy_transactions()

# Build a DataFrame: one row per customer, item list + metadata
transactions_df = pd.DataFrame({
    "customer_id": [f"CUST_{i:03d}" for i in range(len(raw_transactions))],
    "items": raw_transactions,
})

# Inject a few nulls / empty baskets to simulate dirty real-world data
dirty_idx = np.random.default_rng(SEED).choice(len(transactions_df), size=3, replace=False)
# .at[] (not .loc[]) is used because we're assigning a single cell to a
# LIST value -- .loc[] tries to broadcast iterables across multiple cells
# and raises "Must have equal len keys and value".
transactions_df.at[dirty_idx[0], "items"] = None
transactions_df.at[dirty_idx[1], "items"] = []
transactions_df.at[dirty_idx[2], "items"] = [np.nan, "Milk", None]

print("Raw (dirty) transaction dataframe — sample:")
transactions_df.head(10)


### 2.2 Clean Nulls / Empty Baskets

In [ ]:
# ============================================================
# 2.2  Clean the transaction data
#      - Drop rows where items is None
#      - Drop rows with empty baskets
#      - Strip stray None/NaN entries WITHIN a basket list
# ============================================================
def clean_basket(items):
    if items is None or (isinstance(items, float) and pd.isna(items)):
        return None
    cleaned = [i for i in items if isinstance(i, str) and i.strip() != ""]
    return cleaned if len(cleaned) > 0 else None

transactions_df["items_clean"] = transactions_df["items"].apply(clean_basket)

before = len(transactions_df)
transactions_df = transactions_df.dropna(subset=["items_clean"]).reset_index(drop=True)
after = len(transactions_df)

print(f"Dropped {before - after} null/empty transaction(s). Remaining: {after}")
transactions_df[["customer_id", "items_clean"]].head(10)


### 2.3 One-Hot Encode with `mlxtend.preprocessing.TransactionEncoder`

In [ ]:
# ============================================================
# 2.3  Transform basket lists -> binary standard matrix
#      TransactionEncoder expects a LIST OF LISTS of item names.
# ============================================================
basket_list = transactions_df["items_clean"].tolist()

te = TransactionEncoder()
te_array = te.fit(basket_list).transform(basket_list)  # boolean 2D numpy array

encoded_df = pd.DataFrame(te_array, columns=te.columns_)
# Cast True/False -> 1/0 for a clean binary matrix (Apriori/KNN friendly)
encoded_df = encoded_df.astype(int)

print("One-hot encoded transaction matrix shape:", encoded_df.shape)
encoded_df.head(10)


### 2.4 Export Preprocessed Matrix to CSV (for Notebook 03)

In [ ]:
# ============================================================
# 2.4  Persist the encoded matrix so ai_correlation_knn.ipynb can load it
# ============================================================
OUTPUT_CSV = "transactions_encoded.csv"
encoded_df.to_csv(OUTPUT_CSV, index=False)

print(f"Saved binary transaction matrix -> {OUTPUT_CSV}")
print(f"{len(encoded_df)} transactions x {len(encoded_df.columns)} unique products")

# In Colab, download it locally if needed:
# from google.colab import files
# files.download(OUTPUT_CSV)


---
## ✅ Notebook 01 Summary

| Pipeline | Output | Consumed by |
|---|---|---|
| Vision | Resized (640×640) + normalized + augmented image, displayed inline | Feeds intuition for Notebook 02's training data pipeline |
| AI | `transactions_encoded.csv` — binary one-hot transaction matrix | Notebook 03 (`ai_correlation_knn.ipynb`) |

**Next:** hand off to the Vision Lead (`ivp_yolov8_model.ipynb`) and AI Lead (`ai_correlation_knn.ipynb`).
